# SBP verification

Compare limited-line reconstruction with a full bead-pull grid. Default alignment uses the full-grid reference. [Calculation](README.md#calculation).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from src import sbp_verification as sbp
from src import verification_plots as plots

## 1. Measurement files

Place full-grid HDF5 files in `DATA_DIR`. Defaults select nine bins near 19.5 GHz; gating uses the full sweep.

In [ ]:
DATA_DIR = Path("data/full_bead_pull")
OUTPUT_DIR = Path("outputs/sbp_verification")
LINE_COUNT = 3
Z_SLICE_COUNT = 1
GAP_COUNT = 4
FREQUENCY_INDICES = np.arange(230, 271, 5)
REFERENCE_ALIGNMENT = True
TIME_GATING = True
N_WORKERS = 1

gaps = sbp.discover_gaps(DATA_DIR)
for gap in gaps:
    print(f"Gap {gap.rank}: {gap.name}, acquisition z = {gap.z_min_mm:g}–{gap.z_max_mm:g} mm")

## 2. Reconstruct

Original line/plane selection, complex Gaussian fit and finite-bead field conversion.

In [ ]:
for gap in gaps[:GAP_COUNT]:
    indices = sbp.select_gap_z_ixs(gap, Z_SLICE_COUNT)
    print(f"Gap {gap.rank}: array indices {indices}, acquisition z [mm] = "
          f"{np.asarray(gap.z_positions_m)[list(indices)] * 1e3}")

result = sbp.verify_scan(
    gaps, line_count=LINE_COUNT, z_slice_count=Z_SLICE_COUNT,
    gap_count=GAP_COUNT, frequency_indices=FREQUENCY_INDICES,
    reference_alignment=REFERENCE_ALIGNMENT, time_gating=TIME_GATING,
    n_workers=N_WORKERS,
)
result["metrics"]

## 3. Transverse fields

White dots: retained samples. Residual: complex difference / full-grid RMS.

In [ ]:
example = result["example"]
f_index = int(np.argmin(abs(result["frequencies"] - 19.5e9)))
field_figure = plots.plot_fields(example, frequency_index=f_index)
plt.show()

## 4. Transverse integrals

$E_T=\int E_{\mathrm{conv}}\,dA$, in V m. The Gaussian supplies the unmeasured transverse shape.

In [ ]:
integral_figure = plots.plot_integrals(result, frequency_index=f_index)
error_figure = plots.plot_integral_error(result)
plt.show()

## 5. Save results

ET arrays use `(z, frequency)`; z retains the acquisition-relative coordinate.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sbp.save_results(result, OUTPUT_DIR / "ET_results.npz")
field_figure.savefig(OUTPUT_DIR / "fields.png", dpi=150)
integral_figure.savefig(OUTPUT_DIR / "transverse_integrals.png", dpi=150)
error_figure.savefig(OUTPUT_DIR / "integral_error.png", dpi=150)

## Other configurations

Change `LINE_COUNT`, `Z_SLICE_COUNT` or `GAP_COUNT`, then rerun from step 2.